In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [54]:
# latest most actual data
df_orig = pd.read_csv('data/128k.csv', index_col=0)

a_weights = pd.read_excel('experiments/hour_mean_14k.xlsx', index_col='hour_of_day')
# b_weights = pd.read_excel('experiments/hour_mean_128k.xlsx')
# c_weights = pd.read_excel('experiments/hour_median_128k.xlsx')

In [55]:
y_features = ['hour' + str(idx) for idx in range(48)]
x_features = ['hour_of_day', 'sum_costs']

df = df_orig.copy()
df[y_features] = df[y_features].divide(df['sum_costs'], axis=0)
df.head()

,day_of_week,hour0,hour1,hour10,hour11,hour12,hour13,hour14,hour15,hour16,...,hour45,hour46,hour47,hour5,hour6,hour7,hour8,hour9,hour_of_day,sum_costs
1,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11,164.597542
2,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,330.038593
3,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9,360.073602
4,5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,145.260000
5,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,617.733559


In [72]:
# random split
# df_train, df_test = train_test_split(df, test_size=0.1, random_state=42)

# true split by time from dimon
df_train = pd.read_csv('data/train.csv', index_col=[0])
df_test = pd.read_csv('data/control.csv', index_col=[0])
df_train[y_features] = df_train[y_features].divide(df_train['sum_costs'], axis=0)
df_test[y_features] = df_test[y_features].divide(df_test['sum_costs'], axis=0)


print(f'train shape: {df_train.shape}')
print(f'test shape : {df_test.shape}')

print(f'a_weights shape: {a_weights.shape}')
# print(f'b_weights shape: {b_weights.shape}')
# print(f'c_weights shape: {c_weights.shape}')

train shape: (59012, 51)
test shape : (62313, 51)
a_weights shape: (24, 48)


In [73]:
df_train.head()

,day_of_week,hour0,hour1,hour10,hour11,hour12,hour13,hour14,hour15,hour16,...,hour45,hour46,hour47,hour5,hour6,hour7,hour8,hour9,hour_of_day,sum_costs
1,4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15,118.371203
2,7,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,97.601797
3,6,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,277.786195
4,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11,228.452593
5,5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,88.425508


In [74]:
from src.metrics import compute_metrics


hour_mean = df_train.groupby(['hour_of_day'])[y_features].mean()
hour_median = df_train.groupby(['hour_of_day'])[y_features].median()

a_pred = df_test.apply(lambda row: a_weights.loc[row['hour_of_day']], axis=1)
hour_mean_pred = df_test.apply(lambda row: hour_mean.loc[row['hour_of_day']], axis=1)
hour_median_pred = df_test.apply(lambda row: hour_median.loc[row['hour_of_day']], axis=1)

print('a:       ', compute_metrics(a_pred.to_numpy(), df_test[y_features].to_numpy())['hellinger'])
print('b mean:  ', compute_metrics(hour_mean_pred.to_numpy(), df_test[y_features].to_numpy())['hellinger'])
print('c median:', compute_metrics(hour_median_pred.to_numpy(), df_test[y_features].to_numpy())['hellinger'])

a:        0.5815146630252637
b mean:   0.6640139041278063
c median: 0.7132228467046922
